# Temporal Mamba Hyperparameter Tuning on `tgbl`

This notebook builds on `exp.run_temporal_tgbl` and turns the same training path into a resumable, resource-aware hyperparameter search workflow for `tgbl-*` datasets.

It is designed to stay efficient by:
- shortlisting only a few snapshot windows from cheap timestamp estimates
- searching first on smaller edge caps and shorter schedules
- skipping test evaluation during the search phase
- using Optuna pruning and early stopping
- promoting only the best few proxy configs to a larger rerank budget
- doing one final clean retrain only after that reranking step


## Environment notes

This notebook expects the same environment as `exp.run_temporal_tgbl`, plus `optuna` for the study loop.

If `optuna` is missing in your kernel, run:

```python
%pip install optuna
```


In [1]:
import sys
print(sys.executable)


/home/zhuowez1/miniconda3/envs/nsd/bin/python


In [2]:
import json
import random
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'exp').exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / 'exp').exists(), 'Run this notebook from the repo root or notebooks/.'

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import exp.temporal_tgbl_studies as tgbl_studies

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_colwidth', 120)
torch.set_printoptions(sci_mode=False)


def _require_optuna():
    try:
        import optuna
    except ImportError as exc:
        raise RuntimeError(
            'This notebook requires Optuna, but the active kernel cannot import it. '
            f'Current interpreter: {sys.executable}. '
            'Install Optuna into this exact interpreter or switch the notebook kernel, then rerun the setup cells.'
        ) from exc
    return optuna


def _seed_everything(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


/home/zhuowez1/miniconda3/envs/nsd/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Tuning knobs

The defaults below are intentionally optimized for search efficiency first.
Start with the proxy search, then let the notebook promote only the strongest configurations to roomier budgets.


In [3]:
DATASET_NAME = 'tgbl-wiki-v2'
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
BASE_SEED = 43

SEARCH_SPLIT_CAPS = {
    'train': 1024,
    'val': 128,
    'test': 128,
}
PROMOTION_SPLIT_CAPS = {
    'train': 2048,
    'val': 256,
    'test': 256,
}
FINAL_SPLIT_CAPS = {
    'train': 4096,
    'val': 512,
    'test': 512,
}

SNAPSHOT_TUNING_SPLIT = 'train'
SNAPSHOT_WINDOW_CANDIDATES = None
MAX_SEARCH_WINDOWS = 4
TARGET_MAX_SEARCH_SNAPSHOTS = 96
TARGET_MAX_MEAN_EDGES = 256

STUDY_NAME = f'temporal-mamba-tgbl-{DATASET_NAME}'
STUDY_DIR = REPO_ROOT / 'notebooks' / 'artifacts'
STUDY_PATH = STUDY_DIR / f'{STUDY_NAME}.db'

SEARCH_TRIALS = 24
SEARCH_TIMEOUT_MINUTES = None
SEARCH_EPOCHS = 24
SEARCH_EVAL_EVERY = 4
SEARCH_PATIENCE = 8
SEARCH_MIN_EPOCHS_BEFORE_STOPPING = 8
SEARCH_EVALUATE_TEST = False

PROMOTION_TOP_K = 5
PROMOTION_EPOCHS = 40
PROMOTION_EVAL_EVERY = 4
PROMOTION_PATIENCE = 12
PROMOTION_MIN_EPOCHS_BEFORE_STOPPING = 12
PROMOTION_EVALUATE_TEST = True

FINAL_EPOCHS = 100
FINAL_EVAL_EVERY = 2
FINAL_PATIENCE = 20
FINAL_MIN_EPOCHS_BEFORE_STOPPING = 24
FINAL_SKIP_TRAIN_EVAL = True
FINAL_EVALUATE_TEST = True

print({
    'dataset': DATASET_NAME,
    'device': str(DEVICE),
    'study_path': str(STUDY_PATH),
    'search_caps': SEARCH_SPLIT_CAPS,
    'promotion_caps': PROMOTION_SPLIT_CAPS,
    'final_caps': FINAL_SPLIT_CAPS,
})


{'dataset': 'tgbl-wiki-v2', 'device': 'cuda:0', 'study_path': '/home/zhuowez1/project/neural-sheaf-diffusion/notebooks/artifacts/temporal-mamba-tgbl-tgbl-wiki-v2.db', 'search_caps': {'train': 1024, 'val': 128, 'test': 128}, 'promotion_caps': {'train': 2048, 'val': 256, 'test': 256}, 'final_caps': {'train': 4096, 'val': 512, 'test': 512}}


## 1. Load the `tgbl` dataset and cache candidate snapshot bundles

We first estimate which snapshot windows are affordable, then build and cache only a small shortlist for the search phase.


In [4]:
_seed_everything(BASE_SEED)

search_context = tgbl_studies.prepare_temporal_experiment_context(
    DATASET_NAME,
    SEARCH_SPLIT_CAPS,
    device=DEVICE,
    seed=BASE_SEED,
)
metric_name = search_context.metric_name
split_edge_ids = search_context.split_edge_ids

if SNAPSHOT_TUNING_SPLIT not in split_edge_ids:
    raise ValueError(f'SNAPSHOT_TUNING_SPLIT must be one of {sorted(split_edge_ids)}')

candidate_windows = tgbl_studies.make_window_candidates(
    search_context.temporal_data.t,
    split_edge_ids[SNAPSHOT_TUNING_SPLIT],
    manual_candidates=SNAPSHOT_WINDOW_CANDIDATES,
)
snapshot_window_estimates = tgbl_studies.estimate_time_window_grid(
    search_context.temporal_data,
    split_edge_ids[SNAPSHOT_TUNING_SPLIT],
    SEARCH_SPLIT_CAPS[SNAPSHOT_TUNING_SPLIT],
    candidate_windows,
)
snapshot_window_recommendation = tgbl_studies.recommend_time_window(
    snapshot_window_estimates,
    target_max_snapshots=TARGET_MAX_SEARCH_SNAPSHOTS,
    target_max_mean_edges=TARGET_MAX_MEAN_EDGES,
)
search_windows = tgbl_studies.choose_search_windows(
    snapshot_window_estimates,
    recommendation=snapshot_window_recommendation,
    max_windows=MAX_SEARCH_WINDOWS,
)

for time_window in search_windows:
    search_context.get_snapshot_bundle(time_window)

display(Markdown(
    f'**Split source:** `{search_context.split_source}`  '
    f'**Metric:** `{metric_name}`  '
    f'**Destination vocabulary size:** `{search_context.destination_spec["size"]}`  '
    f'**Search windows:** `{search_windows}`'
))
display(snapshot_window_estimates[[
    'window_label',
    'edge_prefix',
    'estimated_snapshots',
    'estimated_mean_edges',
    'estimated_max_edges',
    'compression_ratio',
    'timestamp_start',
    'timestamp_end',
]])
if snapshot_window_recommendation is not None:
    display(Markdown(
        f"**Recommended proxy window:** `{snapshot_window_recommendation['window_label']}` "
        f"with about **{int(snapshot_window_recommendation['estimated_snapshots'])}** snapshots."
    ))
display(search_context.snapshot_table())


**Split source:** `official_tgb_masks`  **Metric:** `mrr`  **Destination vocabulary size:** `1000`  **Search windows:** `[None, 708, 36316, 134930]`

,window_label,edge_prefix,estimated_snapshots,estimated_mean_edges,estimated_max_edges,compression_ratio,timestamp_start,timestamp_end
0,exact timestamps,1024,1004,1.019920,3,1.000000,0,28546
1,1,1024,1004,1.019920,3,1.000000,0,28546
2,4,1024,952,1.075630,3,0.948207,0,28546
3,14,1024,807,1.268897,4,0.803785,0,28546
4,51,1024,462,2.216450,8,0.460159,0,28546
5,191,1024,149,6.872483,17,0.148406,0,28546
6,708,1024,41,24.975609,43,0.040837,0,28546
7,2631,1024,11,93.090912,122,0.010956,0,28546
8,9774,1024,3,341.333344,407,0.002988,0,28546
9,36316,1024,1,1024.000000,1024,0.000996,0,28546


**Recommended proxy window:** `708` with about **41** snapshots.

,time_window,window_label,train_edges,train_snapshots,train_mean_edges,val_edges,val_snapshots,test_edges,test_snapshots
0,NaN,exact timestamps,1024,1004,1.01992,128,126,128,126
1,708.0,708,1024,41,24.97561,128,3,128,2
2,36316.0,36316,1024,1,1024.00000,128,1,128,1
3,134930.0,134930,1024,1,1024.00000,128,1,128,1


## 2. Define the Optuna study

The objective samples model and training hyperparameters, trains on the cached proxy snapshot bundle, and optimizes validation MRR only.


In [5]:
optuna = _require_optuna()

STUDY_DIR.mkdir(parents=True, exist_ok=True)
storage_url = f'sqlite:///{STUDY_PATH}'
sampler = optuna.samplers.TPESampler(seed=BASE_SEED, multivariate=True)
pruner = optuna.pruners.MedianPruner(
    n_startup_trials=6,
    n_warmup_steps=SEARCH_MIN_EPOCHS_BEFORE_STOPPING,
    interval_steps=SEARCH_EVAL_EVERY,
)
study = optuna.create_study(
    study_name=STUDY_NAME,
    direction='maximize',
    sampler=sampler,
    pruner=pruner,
    storage=storage_url,
    load_if_exists=True,
)


def objective(trial):
    config = tgbl_studies.sample_trial_config(
        trial,
        search_windows=search_windows,
        search_epochs=SEARCH_EPOCHS,
        eval_every=SEARCH_EVAL_EVERY,
        patience=SEARCH_PATIENCE,
        min_epochs_before_stopping=SEARCH_MIN_EPOCHS_BEFORE_STOPPING,
        evaluate_test=SEARCH_EVALUATE_TEST,
    )
    snapshot_bundle = search_context.get_snapshot_bundle(config['time_window'])
    result = tgbl_studies.train_single_config(
        context=search_context,
        snapshot_bundle=snapshot_bundle,
        config=config,
        seed=BASE_SEED + trial.number,
        keep_history=False,
        trial=trial,
    )
    trial.set_user_attr('metric_name', result['metric_name'])
    trial.set_user_attr('best_epoch', result['best_epoch'])
    trial.set_user_attr('search_best_test_metric', result['search_best_test_metric'])
    trial.set_user_attr('config_json', json.dumps(config, sort_keys=True))
    return result['best_val_metric']


completed_trials = sum(
    trial.state == optuna.trial.TrialState.COMPLETE
    for trial in study.trials
)
remaining_trials = max(SEARCH_TRIALS - completed_trials, 0)

print({
    'study_name': STUDY_NAME,
    'storage_url': storage_url,
    'completed_trials': completed_trials,
    'remaining_trials': remaining_trials,
    'search_trials_target': SEARCH_TRIALS,
})


/home/zhuowez1/miniconda3/envs/nsd/lib/python3.9/site-packages/optuna/_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(
[I 2026-05-01 08:28:13,058] Using an existing study with name 'temporal-mamba-tgbl-tgbl-wiki-v2' instead of creating a new one.


{'study_name': 'temporal-mamba-tgbl-tgbl-wiki-v2', 'storage_url': 'sqlite:////home/zhuowez1/project/neural-sheaf-diffusion/notebooks/artifacts/temporal-mamba-tgbl-tgbl-wiki-v2.db', 'completed_trials': 15, 'remaining_trials': 9, 'search_trials_target': 24}


## 3. Run or resume the proxy study

Re-running this cell keeps the same study and adds more completed trials until the target is reached.


In [ ]:
# helps resuming experiment without restarting kernel. 
import importlib
import exp.temporal_tgbl_studies as tgbl_studies
importlib.reload(tgbl_studies)

In [ ]:
if remaining_trials > 0:
    optimize_kwargs = {
        'func': objective,
        'n_trials': remaining_trials,
        'gc_after_trial': True,
    }
    if SEARCH_TIMEOUT_MINUTES is not None:
        optimize_kwargs['timeout'] = int(SEARCH_TIMEOUT_MINUTES * 60)
    study.optimize(**optimize_kwargs)
else:
    print('Target number of completed trials already reached; skipping optimize().')

best_trial = study.best_trial
display(Markdown(
    f"**Best validation {metric_name}:** `{best_trial.value:.4f}` from trial `{best_trial.number}`"
))
display(tgbl_studies.config_frame(json.loads(best_trial.user_attrs['config_json'])))


## 4. Inspect the proxy search results


In [ ]:
proxy_completed_df, proxy_top_configs = tgbl_studies.top_completed_trials(study, top_k=PROMOTION_TOP_K)
display(proxy_completed_df.head(10))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
if not proxy_completed_df.empty:
    axes[0].plot(proxy_completed_df.index + 1, proxy_completed_df['value'], marker='o', linewidth=1)
    axes[0].set_title(f'Completed trial validation {metric_name}')
    axes[0].set_xlabel('Ranked completed trial')
    axes[0].set_ylabel(metric_name)

    running_best = proxy_completed_df['value'].cummax()
    axes[1].plot(range(1, len(running_best) + 1), running_best, marker='o', linewidth=1)
    axes[1].set_title('Best-so-far validation metric')
    axes[1].set_xlabel('Completed trial')
    axes[1].set_ylabel(metric_name)
else:
    axes[0].text(0.5, 0.5, 'No completed trials yet', ha='center', va='center')
    axes[1].text(0.5, 0.5, 'No completed trials yet', ha='center', va='center')

plt.tight_layout()
plt.show()

try:
    importances = optuna.importance.get_param_importances(study)
    importance_df = pd.DataFrame(importances.items(), columns=['parameter', 'importance'])
    display(importance_df)
except Exception as exc:
    print(f'Parameter importance unavailable: {exc}')


## 5. Promote the top proxy configs to a larger rerank budget

This stage spends a bit more compute only on the strongest proxy configurations.


In [ ]:
promotion_configs = [
    tgbl_studies.merge_final_config(
        config,
        final_epochs=PROMOTION_EPOCHS,
        eval_every=PROMOTION_EVAL_EVERY,
        patience=PROMOTION_PATIENCE,
        min_epochs_before_stopping=PROMOTION_MIN_EPOCHS_BEFORE_STOPPING,
        skip_train_eval=True,
        evaluate_test=PROMOTION_EVALUATE_TEST,
    )
    for config in proxy_top_configs
]

promotion_windows = sorted({tgbl_studies._canonical_time_window(config.get('time_window')) for config in promotion_configs}, key=lambda x: -1 if x is None else x)
promotion_context = tgbl_studies.prepare_temporal_experiment_context(
    DATASET_NAME,
    PROMOTION_SPLIT_CAPS,
    device=DEVICE,
    seed=BASE_SEED,
    preload_time_windows=promotion_windows,
)

display(promotion_context.snapshot_table())

rerank_df = tgbl_studies.rerank_configs(
    promotion_context,
    promotion_configs,
    seed_base=BASE_SEED,
    progress=True,
)
display(rerank_df)


## 6. Retrain the best promoted configuration and evaluate it cleanly

This final pass rebuilds the snapshot bundle with `FINAL_SPLIT_CAPS`, then trains longer with a roomier patience budget.


In [ ]:
if rerank_df.empty:
    raise RuntimeError('No promoted configurations were available. Run the study first.')

best_rerank_config = json.loads(rerank_df.iloc[0]['config_json'])
final_config = tgbl_studies.merge_final_config(
    best_rerank_config,
    final_epochs=FINAL_EPOCHS,
    eval_every=FINAL_EVAL_EVERY,
    patience=FINAL_PATIENCE,
    min_epochs_before_stopping=FINAL_MIN_EPOCHS_BEFORE_STOPPING,
    skip_train_eval=FINAL_SKIP_TRAIN_EVAL,
    evaluate_test=FINAL_EVALUATE_TEST,
)

final_context = tgbl_studies.prepare_temporal_experiment_context(
    DATASET_NAME,
    FINAL_SPLIT_CAPS,
    device=DEVICE,
    seed=BASE_SEED,
    preload_time_windows=[final_config.get('time_window')],
)
final_snapshot_bundle = final_context.get_snapshot_bundle(final_config['time_window'])
display(final_context.snapshot_table())

final_result = tgbl_studies.train_single_config(
    context=final_context,
    snapshot_bundle=final_snapshot_bundle,
    config=final_config,
    seed=BASE_SEED,
    keep_history=True,
    trial=None,
)

final_summary = pd.DataFrame([
    {
        'split': 'train',
        metric_name: final_result['best_train_metric'],
        'loss': final_result['best_train_loss'],
        'snapshots': len(final_snapshot_bundle['train_snapshots']),
    },
    {
        'split': 'val',
        metric_name: final_result['best_val_metric'],
        'loss': final_result['best_val_loss'],
        'snapshots': len(final_snapshot_bundle['val_snapshots']),
    },
    {
        'split': 'test',
        metric_name: final_result['best_test_metric'],
        'loss': final_result['best_test_loss'],
        'snapshots': len(final_snapshot_bundle['test_snapshots']),
    },
])
display(tgbl_studies.config_frame(final_config))
display(final_summary)


In [ ]:
history_df = final_result['history'].copy()
val_col = tgbl_studies._find_metric_column(history_df, 'val', metric_name)
test_col = tgbl_studies._find_metric_column(history_df, 'test', metric_name)
eval_columns = [col for col in [val_col, test_col, 'val_loss', 'test_loss'] if col is not None and col in history_df.columns]
if eval_columns:
    eval_history_df = history_df.dropna(subset=eval_columns, how='all').copy()
else:
    eval_history_df = history_df.iloc[0:0].copy()


def _plot_metric(ax, df, column, title):
    ax.set_title(title)
    if column is None or column not in df.columns:
        ax.text(0.5, 0.5, 'Metric column not found in history', ha='center', va='center', transform=ax.transAxes)
        return

    plot_df = df[['epoch', column]].dropna().copy()
    if plot_df.empty:
        ax.text(0.5, 0.5, 'No evaluated data to plot yet', ha='center', va='center', transform=ax.transAxes)
        return

    plot_df.plot(x='epoch', y=column, ax=ax, marker='o', linewidth=1.5, legend=False)


fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
history_df.plot(x='epoch', y='train_loss', ax=axes[0], title='Training loss', legend=False)
_plot_metric(axes[1], eval_history_df, val_col, title=f'Validation {metric_name}')
_plot_metric(axes[2], eval_history_df, test_col, title=f'Test {metric_name}')
for ax in axes:
    ax.set_xlabel('epoch')
plt.tight_layout()
plt.show()


## 7. Save artifacts and reproduce the final run from the command line


In [ ]:
results_dir = REPO_ROOT / 'results' / f'{final_context.dataset_name}_TemporalMambaSheaf_tuning_{int(time.time())}'
results_dir.mkdir(parents=True, exist_ok=True)

snapshot_window_estimates.to_csv(results_dir / 'snapshot_window_estimates.csv', index=False)
proxy_completed_df.to_csv(results_dir / 'proxy_completed_trials.csv', index=False)
rerank_df.to_csv(results_dir / 'promotion_rerank.csv', index=False)
final_summary.to_csv(results_dir / 'final_summary.csv', index=False)
history_df.to_csv(results_dir / 'final_history.csv', index=False)
tgbl_studies.config_frame(final_config).to_csv(results_dir / 'final_config.csv', index=False)

reproduction_command = tgbl_studies.render_reproduction_command(final_config, DATASET_NAME, FINAL_SPLIT_CAPS)
display(Markdown('```bash\n' + reproduction_command + '\n```'))
print(f'Saved tuning artifacts to {results_dir}')
